# ALL CODE

## ===========================================================

### Installs and imports

In [ ]:
# %pip install -q transformers trl peft bitsandbytes openai numpy pandas torch fsspec==2025.3.2
# %pip install -q transformers==4.52.4 trl==0.11.4 peft==0.15.2 bitsandbytes==0.46.0 openai==1.84.0 numpy==2.0.2 pandas==2.2.2 torch==2.8.0 fsspec==2025.3.2

In [1]:
# Standard library imports
import os
import re
import textwrap
from typing import List


# Third-party imports
import numpy as np
import pandas as pd
import torch

# OpenAI API client
from openai import OpenAI

# Custom modules for system prompts and questionnaires
from system_prompts_builder_V2 import generate_all_permutations, CounselorPersonality
# from questionnaires_V3 import get_prompt_eval_questionnaire, parse_questionnaire_response, WAI_SR_LABELS, WAI_SR_ITEMS, WAI_SR_SUBSCALES
from questionnaires_V3 import (
    get_prompt_eval_questionnaire,
    parse_questionnaire_response,
    WAI_SR_LABELS, WAI_SR_ITEMS, WAI_SR_SUBSCALES,
    CSQ8_LABELS, CTRL_LABELS    
)


# Transformers and related libraries for model loading and quantization
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)
from peft import PeftModel
from trl import setup_chat_format

# Progress bar utility
from tqdm import tqdm

# Additional imports for version info
import transformers
import trl
import peft
import bitsandbytes
import openai
import fsspec

# Print version information for reproducibility and debugging
print(f"torch version: {torch.__version__}")
print(f"transformers version: {transformers.__version__}")
print(f"trl version: {trl.__version__}")
print(f"peft version: {peft.__version__}")
print(f"bitsandbytes version: {bitsandbytes.__version__}")
print(f"openai version: {openai.__version__}")
print(f"numpy version: {np.__version__}")
print(f"pandas version: {pd.__version__}")
print(f"fsspec version: {fsspec.__version__}")

# Set device for computation (GPU if available, else CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print(f"cuda version: {torch.version.cuda}")


c:\Users\baruc\Desktop\Preference_Trees\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


torch version: 2.8.0+cu126
transformers version: 4.56.2
trl version: 0.23.0
peft version: 0.17.1
bitsandbytes version: 0.47.0
openai version: 1.109.1
numpy version: 2.3.3
pandas version: 2.3.2
fsspec version: 2025.9.0
Using device: cuda
cuda version: 12.6


c:\Users\baruc\Desktop\Preference_Trees\.venv\Lib\site-packages\torch\cuda\__init__.py:283: UserWarning: 
    Found GPU0 NVIDIA GeForce RTX 5070 Ti Laptop GPU which is of cuda capability 12.0.
    Minimum and Maximum cuda capability supported by this version of PyTorch is
    (6.1) - (9.0)
    
  warnings.warn(
c:\Users\baruc\Desktop\Preference_Trees\.venv\Lib\site-packages\torch\cuda\__init__.py:304: UserWarning: 
    Please install PyTorch with a following CUDA
    configurations:  12.8 12.9 following instructions at
    https://pytorch.org/get-started/locally/
    
  warnings.warn(matched_cuda_warn.format(matched_arches))
c:\Users\baruc\Desktop\Preference_Trees\.venv\Lib\site-packages\torch\cuda\__init__.py:326: UserWarning: 
NVIDIA GeForce RTX 5070 Ti Laptop GPU with CUDA capability sm_120 is not compatible with the current PyTorch installation.
The current PyTorch install supports CUDA capabilities sm_61 sm_70 sm_75 sm_80 sm_86 sm_90.
If you want to use the NVIDIA GeForce RTX 5070

### Load OpenAI client to access the API

In [2]:
import os
import sys
# Walk up from CWD to find the workspace root (depth-independent).
_HELPER_FILES = ['openai_key.txt', 'HF_key.txt']
_cur = os.path.abspath(os.getcwd())
_WORKSPACE_ROOT = None
for _ in range(8):
    if all(os.path.exists(os.path.join(_cur, _hf)) for _hf in _HELPER_FILES):
        _WORKSPACE_ROOT = _cur
        break
    _parent = os.path.dirname(_cur)
    if _parent == _cur:
        break
    _cur = _parent
if _WORKSPACE_ROOT is None:
    raise RuntimeError('Could not locate workspace root with key files')
if _WORKSPACE_ROOT not in sys.path:
    sys.path.insert(0, _WORKSPACE_ROOT)
# OpenAI_API_KEY = "Put your key here"

# Load OpenAI API key from a file
OpenAI_API_KEY = open(os.path.join(_WORKSPACE_ROOT, "openai_key.txt"), "r").read().strip()

# Create a client instance with your API key
client = OpenAI(
    api_key=OpenAI_API_KEY
)


### Load the therapist Tokenizer and Model

Login to huggingface

In [ ]:
import os
import sys
# Walk up from CWD to find the workspace root (depth-independent).
_HELPER_FILES = ['openai_key.txt', 'HF_key.txt']
_cur = os.path.abspath(os.getcwd())
_WORKSPACE_ROOT = None
for _ in range(8):
    if all(os.path.exists(os.path.join(_cur, _hf)) for _hf in _HELPER_FILES):
        _WORKSPACE_ROOT = _cur
        break
    _parent = os.path.dirname(_cur)
    if _parent == _cur:
        break
    _cur = _parent
if _WORKSPACE_ROOT is None:
    raise RuntimeError('Could not locate workspace root with key files')
if _WORKSPACE_ROOT not in sys.path:
    sys.path.insert(0, _WORKSPACE_ROOT)
# For local use (uncomment if running locally):
from huggingface_hub import login
hf_token = open(os.path.join(_WORKSPACE_ROOT, "HF_key.txt"), "r").read().strip()
login(token=hf_token)  # Log in to the Hugging Face hub (required for private datasets/models)

# # For Colab use:
# from huggingface_hub import login
# from google.colab import userdata
# hf_token = userdata.get("huggingface")
# login(token=hf_token)

Load Model and Tokenizer

In [ ]:
# Choose the model ID for the therapist model
therapist_model_id = "meta-llama/Llama-3.2-1B"


# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(therapist_model_id, trust_remote_code=True, device_map=device)
# set chat template
tokenizer.chat_template = "{% if not add_generation_prompt is defined %}{% set add_generation_prompt = false %}{% endif %}{% for message in messages %}{{'<|im_start|>' + message['role'] + '\n' + message['content'] + '<|im_end|>' + '\n'}}{% endfor %}{% if add_generation_prompt %}{{ '<|im_start|>assistant\n' }}{% endif %}"
# set padding token and padding side
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"  # Fix weird overflow issue with fp16 training

############### Test the tokenizer with a sample message ###############
print("Spacial tokens: ", tokenizer.special_tokens_map)
test_massage = [{'role': 'system', 'content': 'you are a robot'},
                {'role': 'user', 'content': 'Hello, how is it to be a robot?'},
                {'role': 'assistant', 'content': 'It is great to be a robot.'},
                {'role': 'user', 'content': 'cool.'}]

chat = tokenizer.apply_chat_template(test_massage, tokenize=False, add_generation_prompt=True)
print(chat)


In [ ]:
# Set quantization config (to save memory)
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4"
)

# Load model, quantized
therapist_model = AutoModelForCausalLM.from_pretrained(
    therapist_model_id,
    quantization_config=quantization_config,
    # device_map=device,
    device_map="auto",
    trust_remote_code=True,
)

# therapist_model.config.use_cache = False

### Choose LookAhead, Oracle and add Adapters

choose LookAhead and Version

In [ ]:
lookAhead = 5
version = 0
oracle_name = "WAI" # Options: "WAI", "CSQ-8", "CTRL"
final_column_name = "WAI_TotalMean" # Options: "WAI_TotalMean", "CSQ8_Mean", "CTRL_Mean"
num_to_place = {0: "base", 1: "first", 2: "second", 3: "third", 4: "fourth", 5: "fifth", 6: "sixth", 7: "seventh"}
therapist_adapter_id = "Llama-3.2-1B-hf"

########################################### New Adapters V3 ############################################
adapter_ids = [
    f"LBK95/{therapist_adapter_id}-DPO_V3-{oracle_name}-LookAhead-{lookAhead}_TTree1.2_TT0.9_TP0.7_TE0.1_V{i+1}"
    for i in range(7)
]
########################################### New Adapters V3 ############################################

display(adapter_ids)

if oracle_name == "WAI":
    questionnaire_id = 3
elif oracle_name == "CSQ-8":
    questionnaire_id = 4
elif oracle_name == "CTRL":
    questionnaire_id = 5
else:
    questionnaire_id = -1

['LBK95/Llama-3.2-1B-hf-DPO_V3-WAI-LookAhead-0_TTree1.2_TT0.9_TP0.7_TE0.1_V1',
 'LBK95/Llama-3.2-1B-hf-DPO_V3-WAI-LookAhead-0_TTree1.2_TT0.9_TP0.7_TE0.1_V2',
 'LBK95/Llama-3.2-1B-hf-DPO_V3-WAI-LookAhead-0_TTree1.2_TT0.9_TP0.7_TE0.1_V3',
 'LBK95/Llama-3.2-1B-hf-DPO_V3-WAI-LookAhead-0_TTree1.2_TT0.9_TP0.7_TE0.1_V4',
 'LBK95/Llama-3.2-1B-hf-DPO_V3-WAI-LookAhead-0_TTree1.2_TT0.9_TP0.7_TE0.1_V5',
 'LBK95/Llama-3.2-1B-hf-DPO_V3-WAI-LookAhead-0_TTree1.2_TT0.9_TP0.7_TE0.1_V6',
 'LBK95/Llama-3.2-1B-hf-DPO_V3-WAI-LookAhead-0_TTree1.2_TT0.9_TP0.7_TE0.1_V7']

Load and merge Adapters

In [ ]:
# add the adapters to the model
for i in range(0, version):
    print("Loading model with adapter ", i+1)
    adapter_id = adapter_ids[i]
    therapist_model = PeftModel.from_pretrained(therapist_model, adapter_id)
    therapist_model = therapist_model.merge_and_unload()
    print(f"Model loaded with {num_to_place[i]} adapter")
    print("Adapter ID: ", adapter_id)

print()
print(f"Model loaded with {version} adapters and {lookAhead} lookAhead")

## ===========================================================

### Methods to Simulate a conversation between a Therapist and a Patient.

Helper Methods

In [ ]:
def reconstruct_conversation_text(utterances: List[str]) -> str:
    return "\n".join([f"[{'THERAPIST' if i % 2 == 0 else 'PATIENT'}]: {utt}" for i, utt in enumerate(utterances)])


def print_conversation(conversation, max_width=80):
    """
    Print the conversation in a formatted way with roles labeled as [THERAPIST] and [PATIENT].
    Args:
        conversation (list): List of message strings, alternating between therapist and patient.
        max_width (int): Maximum width for text wrapping.
    """
    for i, message in enumerate(conversation):
        role = "[THERAPIST]" if i % 2 == 0 else "[PATIENT]"
        print(f"{role}: \n{textwrap.fill(message, width=max_width)} \n")

def call_openai(prompt: str, client: OpenAI = client, model: str = "gpt-4o-mini-2024-07-18", temperature: float = 0.1) -> str:
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=temperature,
        seed=42
    )
    response = response.choices[0].message.content
    return response

def generate_patient_response(client, model_id, messages_Patient_assist, max_tokens, temperature):
    """
    Generate a response from the patient using the OpenAI API (or similar API).
    Args:
        client (OpenAI): OpenAI API client instance.
        model_id (str): Model ID to use for generating responses.
        messages_Patient_assist (list): List of messages for the patient assistant.
        max_tokens (int): Maximum number of tokens to generate.
        temperature (float): Temperature for response generation.
    """
    response = client.chat.completions.create(
        model=model_id,
        messages=messages_Patient_assist,
        max_tokens=max_tokens,
        temperature=temperature,
        seed=42
    )
    response_content = response.choices[0].message.content
    return response_content


def generate_therapist_responses(therapist_model, therapis_tokenizer, messages_Therapist_assist, max_tokens, temperature, num_responses=2, max_attempts=10):
    """
    Generate multiple responses from the therapist model.
    Args:
        therapist_model (AutoModelForCausalLM): The therapist model to generate responses.
        therapis_tokenizer (AutoTokenizer): The tokenizer for the therapist model.
        messages_Therapist_assist (list): List of messages for the therapist assistant.
        max_tokens (int): Maximum number of tokens to generate for each response.
        temperature (float): Temperature for response generation.
        num_responses (int): Number of valid responses to generate.
        max_attempts (int): Maximum number of attempts to generate valid responses.
    Returns:
        list: A list of valid responses generated by the therapist model.
    """
    valid_responses = []
    attempts = max_attempts

    while len(valid_responses) < num_responses and attempts > 0:
        attempts -= 1
        # Apply the chat template and encode the prompt
        prompt = therapis_tokenizer.apply_chat_template(messages_Therapist_assist, tokenize=False, add_generation_prompt=True)
        input_ids = therapis_tokenizer.encode(prompt, return_tensors="pt", add_special_tokens=False).to(therapist_model.device)

        # Generate responses for the therapist
        with torch.no_grad():
            responses = therapist_model.generate(
                input_ids,
                do_sample=True,
                max_new_tokens=max_tokens,
                pad_token_id=therapis_tokenizer.eos_token_id,
                eos_token_id=therapis_tokenizer.eos_token_id,
                temperature=temperature,
                num_return_sequences=num_responses - len(valid_responses),  # Adjust the number of responses needed
                stop_strings=["<|im_end|>"],  # Stop generation at "<|im_end|>"
                tokenizer=therapis_tokenizer
            )

        # Decode responses
        for response in responses:
            decoded_response_original = therapis_tokenizer.decode(response[len(input_ids[0]):], skip_special_tokens=True)
            # get up to <|im_end|> token
            cleaned_response = decoded_response_original.split("<|im_end|>")[0].strip()
            # Check if the response is valid (not empty)
            if cleaned_response and len(cleaned_response) > 0:
                valid_responses.append(cleaned_response)
            else:
                print("Invalid response generated, retrying...")

    if attempts <= 0 and len(valid_responses) < num_responses:
        # raise RuntimeError("Failed to generate the required number of valid responses within the allowed attempts.")
        print("Failed to generate the required number of valid responses within the allowed attempts.")
        return None  # Return None if unable to generate the required number of valid responses
    
    return valid_responses  # Return the required number of valid responses


def handle_session_end(response_content, turn_index):
    """
    Check if the response content contains the 'SESSION ENDED' keyword and handle session termination.

    Args:
        response_content (str): The generated response content to check.
        turn_index (int): The index of the current turn in the conversation (used to determine who ended the session).

    Returns:
        tuple: (session_ended_by, session_ended_explanation, response_content)
            - session_ended_by (str): 'patient' if the patient ended the session, 'therapist' otherwise.
            - session_ended_explanation (str): Explanation or content after the 'SESSION ENDED' keyword.
            - response_content (str): The response content before the 'SESSION ENDED' keyword.

    Raises:
        ValueError: If the 'SESSION ENDED' keyword is not found in the response content.
    """
    session_ended_keyword = "SESSION ENDED"
    idx = response_content.upper().find(session_ended_keyword)

    if idx != -1:
        session_endded_explanation = response_content[idx + len(session_ended_keyword):]
        response_content = response_content[:idx]
        session_endded_by = "patient" if turn_index % 2 == 0 else "therapist"
        print("Response content (SESSION ENDED): ", response_content)
        print("Session ended by: ", session_endded_by)
        return session_endded_by, session_endded_explanation, response_content
    else:
        raise ValueError("SESSION ENDED keyword not found in response content")


def update_conversation(conversation, messages_Patient_assist, messages_Therapist_assist, role_Patient, role_Therapist, response_content):
    """
    Update the conversation and message lists with a new response.

    Args:
        conversation (list): The main conversation list to append the new response to.
        messages_Patient_assist (list): List of messages for the patient assistant, to be updated.
        messages_Therapist_assist (list): List of messages for the therapist assistant, to be updated.
        role_Patient (str): Role label for the patient (e.g., "user" or "assistant").
        role_Therapist (str): Role label for the therapist (e.g., "user" or "assistant").
        response_content (str): The content of the new response to add.
    """
    conversation.append(response_content)
    messages_Patient_assist.append({"role": role_Patient, "content": response_content})
    messages_Therapist_assist.append({"role": role_Therapist, "content": response_content})


def initialize_conversation(system_prompt_therapist, system_prompt_patient, therapist_init_utterance, patient_init_utterance):
    """
    Initialize the conversation and message lists for both therapist and patient assistants.

    Args:
        system_prompt_therapist (str): System prompt for the therapist assistant.
        system_prompt_patient (str): System prompt for the patient assistant.
        therapist_init_utterance (str): Initial utterance from the therapist.
        patient_init_utterance (str): Initial utterance from the patient.

    Returns:
        tuple: (conversation, messages_Patient_assist, messages_Therapist_assist)
            - conversation (list): List containing the initial therapist utterance.
            - messages_Patient_assist (list): Initialized messages for the patient assistant.
            - messages_Therapist_assist (list): Initialized messages for the therapist assistant.
    """
    # Initialize the conversation
    conversation = [therapist_init_utterance]  # Therapist's initial utterance

    # Initialize the messages for the patient as the assistant
    messages_Patient_assist = [
        {"role": "system", "content": system_prompt_patient},
        {"role": "user", "content": therapist_init_utterance}  # Therapist's initial utterance
    ]

    # Initialize the messages for the therapist as the assistant
    messages_Therapist_assist = [
        {"role": "system", "content": system_prompt_therapist},
        {"role": "user", "content": patient_init_utterance},  # Patient's initial utterance (llama model needs a user utterance to start)
        {"role": "assistant", "content": therapist_init_utterance}  # Therapist's initial utterance
    ]

    return conversation, messages_Patient_assist, messages_Therapist_assist


Main Methods

In [5]:
def conversation_loop(
    conversation,
    messages_Patient_assist,
    messages_Therapist_assist,
    therapist_model,
    therapist_tokenizer,
    client,
    model_id="gpt-4o-mini-2024-07-18",
    max_tokens=100,
    num_utterances=6,
    temperature_therapist=0.7,
    temperature_patient=0.7
):
    """
    Simulate a conversation loop between a therapist (local model) and a patient (OpenAI API).

    Args:
        conversation (list): List of conversation utterances.
        messages_Patient_assist (list): Messages for the patient assistant (OpenAI).
        messages_Therapist_assist (list): Messages for the therapist assistant (local model).
        therapist_model: The therapist model (local).
        therapist_tokenizer: Tokenizer for the therapist model.
        client: OpenAI API client.
        model_id (str): Model ID for the patient (OpenAI).
        max_tokens (int): Maximum tokens per response.
        num_utterances (int): Number of utterances to generate.
        temperature_therapist (float): Sampling temperature for therapist.
        temperature_patient (float): Sampling temperature for patient.

    Returns:
        tuple: (conversation, messages_Patient_assist, messages_Therapist_assist, session_endded_by, session_endded_explanation)
    """
    session_endded_by = None
    session_endded_explanation = None

    for i in range(num_utterances):
        if i % 2 == 0:
            # Patient's turn
            role_Patient = "assistant"
            role_Therapist = "user"
            response_content = generate_patient_response(
                client, model_id, messages_Patient_assist, max_tokens, temperature_patient
            )
            print("[Patient]:", response_content)
        else:
            # Therapist's turn
            role_Patient = "user"
            role_Therapist = "assistant"
            responses = generate_therapist_responses(
                therapist_model, therapist_tokenizer, messages_Therapist_assist, max_tokens, temperature_therapist, 1
            )
            if responses is None:
                print("Could not generate a valid response for the therapist, returning None.")
                return None, None, None, None, None
            response_content = responses[0]
            print("-" * 50)
            print("[Therapist]:", response_content)

        # Check if the session has ended
        if "SESSION ENDED" in response_content.upper():
            session_endded_by, session_endded_explanation, response_content = handle_session_end(response_content, i)
            if response_content:
                update_conversation(
                    conversation, messages_Patient_assist, messages_Therapist_assist,
                    role_Patient, role_Therapist, response_content
                )
            break

        # Update the conversation
        update_conversation(
            conversation, messages_Patient_assist, messages_Therapist_assist,
            role_Patient, role_Therapist, response_content
        )

    return conversation, messages_Patient_assist, messages_Therapist_assist, session_endded_by, session_endded_explanation


def synthesize_conversation_therapistModel_patientOpenAI(
    system_prompt_therapist,
    system_prompt_patient,
    therapist_init_utterance,
    patient_init_utterance,
    therapist_model,
    therapis_tokenizer,
    client,
    model_id="gpt-4o-mini-2024-07-18",
    max_tokens=50,
    num_utterances=6,
    temperature_therapist=0.7,
    temperature_patient=0.7
):
    """
    Synthesize a conversation between a therapist (local model) and a patient (OpenAI API).

    Args:
        system_prompt_therapist (str): System prompt for the therapist.
        system_prompt_patient (str): System prompt for the patient.
        therapist_init_utterance (str): Initial utterance from the therapist.
        patient_init_utterance (str): Initial utterance from the patient.
        therapist_model: The therapist model (local).
        therapis_tokenizer: Tokenizer for the therapist model.
        client: OpenAI API client.
        model_id (str): Model ID for the patient (OpenAI).
        max_tokens (int): Maximum tokens per response.
        num_utterances (int): Number of utterances to generate.
        temperature_therapist (float): Sampling temperature for therapist.
        temperature_patient (float): Sampling temperature for patient.

    Returns:
        tuple: (conversation, messages_Patient_assist, messages_Therapist_assist, session_endded_by, session_endded_explanation)
    """
    # Initialize the conversation and message lists
    conversation, messages_Patient_assist, messages_Therapist_assist = initialize_conversation(
        system_prompt_therapist, system_prompt_patient, therapist_init_utterance, patient_init_utterance
    )
    print("Therapist's initial utterance:", therapist_init_utterance)
    # Run the conversation loop
    return conversation_loop(
        conversation, messages_Patient_assist, messages_Therapist_assist,
        therapist_model, therapis_tokenizer, client, model_id,
        max_tokens, num_utterances, temperature_therapist, temperature_patient
    )


### Methods to Evaluate a conversation

In [6]:
def build_wai_sr_row(scores, response, include_notes=False):
    # Build row with all items (NaN if missing)
    row = {k: scores.get(k, np.nan) for k in WAI_SR_LABELS}
    # Subscales = mean over their items (still in 1-5 units)
    for sub_name, items in WAI_SR_SUBSCALES.items():
        vals = [row[i] for i in items if i in row]
        row[f"{sub_name}_Mean"] = float(np.nanmean(vals)) if len(vals) else np.nan
    # Overall means/sums
    item_vals = [row[k] for k in WAI_SR_LABELS]
    row["WAI_TotalMean"] = float(np.nanmean(item_vals))
    row["WAI_TotalSum"]  = float(np.nansum(item_vals))
    # Optionally capture raw rationale
    if include_notes:
        m = re.search(r"\*\*Rationale \(brief\):\*\*([\s\S]+)$", response or "")
        row["WAI_RationaleRaw"] = (m.group(1).strip() if m else "")
    return row


def build_csq8_row(scores: dict, response: str = None):
    row = {k: scores.get(k, np.nan) for k in CSQ8_LABELS}
    item_vals = [row[k] for k in CSQ8_LABELS]
    row["CSQ8_Mean"] = float(np.nanmean(item_vals))
    row["CSQ8_Total"] = float(np.nansum(item_vals))
    return row


def build_ctrl_row(scores: dict, response: str = None, include_notes: bool = False):
    # Ensure all 8 items exist; missing -> np.nan
    row = {k: scores.get(k, np.nan) for k in CTRL_LABELS}

    # Aggregates
    item_vals = [row[k] for k in CTRL_LABELS]
    row["CTRL_Mean"] = float(np.nanmean(item_vals))
    row["CTRL_Total"] = float(np.nansum(item_vals))

    # Optional rationale capture (mirrors WAI logic)
    if include_notes and response is not None:
        m = re.search(r"\*\*Rationale \(brief\):\*\*([\s\S]+)$", response or "")
        row["CTRL_RationaleRaw"] = (m.group(1).strip() if m else "")

    return row



In [7]:
def eval_conversation(
    conversation_str, 
    client, 
    questionnaire=3, 
    model_id="gpt-4o-mini-2024-07-18", 
    eval_temperature=0.2,
    ):
    # Generate the evaluation prompt
    eval_dict = get_prompt_eval_questionnaire(
        questionnaire=questionnaire, 
        conversation=conversation_str
        )

    eval_prompt = eval_dict["prompt"]
    # eval_questions_count = eval_dict["questions_count"]

    # Generate the completion for evaluation
    eval_results = call_openai(
        prompt=eval_prompt, 
        client=client,
        model=model_id, 
        temperature=eval_temperature
        )

    return eval_results, eval_dict


In [8]:
# --- One-convo → DataFrame (+subscales & totals) ---
def evaluate_conversation_and_extract_scores(conversation, model="gpt-4o-mini-2024-07-18", questionnaire_id=3, include_notes=False, print_response=False, eval_temperature=0.1):
    
    # Get WAI-SR evaluation
    response, eval_dict = eval_conversation(
        conversation_str=reconstruct_conversation_text(conversation),
        client=client,
        questionnaire=questionnaire_id,
        model_id=model,
        eval_temperature=eval_temperature,
    )
    if print_response:
        print("\nRaw Response:\n", response)
    
    # Extract scores and validate
    scores = parse_questionnaire_response(response, questionnaire_id)
    num_of_questions = eval_dict["questions_count"]

    if len(scores) != num_of_questions:
        print(f"Warning: Expected {num_of_questions} scores, but got {len(scores)}. returning None.")
        return None

    if questionnaire_id == 3:
        # WAI-SR specific processing
        row = build_wai_sr_row(scores, response, include_notes=include_notes)
    elif questionnaire_id == 4:
        # CSQ-8 specific processing
        row = build_csq8_row(scores, response)
    elif questionnaire_id == 5:
        # CTRL specific processing
        row = build_ctrl_row(scores, response, include_notes=include_notes)
    else:
        row = scores

    return pd.DataFrame([row], columns=list(row.keys()))

# test the function above
# dummy_conversation = [
#     "Hello, how are you feeling today?",
#     "I'm feeling a bit anxious about my upcoming exams.",
#     "ok, i understand."]

# df3 = evaluate_conversation_and_extract_scores(dummy_conversation, print_response=True, include_notes=True, questionnaire_id=3)
# display(df3)
# df4 = evaluate_conversation_and_extract_scores(dummy_conversation, print_response=True, include_notes=False, questionnaire_id=4)
# display(df4)
# df5 = evaluate_conversation_and_extract_scores(dummy_conversation, print_response=True, include_notes=True, questionnaire_id=5)
# display(df5)


### Methods to synthesize conversation trees

Helper Methods

In [9]:
def prepare_lookahead_conversation(conversation, messages_Patient_assist, messages_Therapist_assist, response):
    """
    Prepare new conversation and message lists for look-ahead branching.

    Args:
        conversation (list): Current conversation history.
        messages_Patient_assist (list): Current patient assistant messages.
        messages_Therapist_assist (list): Current therapist assistant messages.
        response (str): Therapist's response to branch on.

    Returns:
        tuple: (conversation_new, messages_Patient_assist_new, messages_Therapist_assist_new)
            - conversation_new (list): Updated conversation with the new response.
            - messages_Patient_assist_new (list): Updated patient assistant messages.
            - messages_Therapist_assist_new (list): Updated therapist assistant messages.
    """
    conversation_new = conversation + [response]
    messages_Patient_assist_new = messages_Patient_assist.copy() + [{"role": "user", "content": response}]
    messages_Therapist_assist_new = messages_Therapist_assist.copy() + [{"role": "assistant", "content": response}]
    return conversation_new, messages_Patient_assist_new, messages_Therapist_assist_new

def determine_winner(responses, scores_one_df, scores_two_df, conversation_one, conversation_two, column_name="WAI_TotalMean"):
    score_one_final = scores_one_df[column_name].values[0]
    score_two_final = scores_two_df[column_name].values[0]


    if score_one_final >= score_two_final:  # Response one is the winner
        return (responses[0], responses[1], scores_one_df, scores_two_df, conversation_one, conversation_two)
    else:  # Response two is the winner
        return (responses[1], responses[0], scores_two_df, scores_one_df, conversation_two, conversation_one)


In [11]:
def synthesize_conversation_tree_lookAhead(
        init_messages_Patient_assist, init_messages_Therapist_assist,
        init_conversation,
        therapist_model,
        therapis_tokenizer,
        client,
        model_id="gpt-4o-mini-2024-07-18",
        max_tokens=100,
        num_utterances=6,
        temperature_tree=1.2,
        temperature_therapist=0.7,
        temperature_patient=0.7,
        temperature_eval=0.2,
        questionnaire_id=3,
        look_ahead=5,
        column_name="WAI_TotalMean",
        include_notes=False,
        print_response=False
    ):
    if init_messages_Patient_assist is None:
        return None, None, None, None, None, None

    preference_data = []
    messages_Patient_assist = init_messages_Patient_assist.copy()
    messages_Therapist_assist = init_messages_Therapist_assist.copy()
    conversation = init_conversation.copy()
    session_endded_by = None
    session_endded_explanation = None

    for i in (range(num_utterances)):
        if i % 2 == 1:  # Patient's turn
            role_Patient = "assistant"
            role_Therapist = "user"
            response_content = generate_patient_response(client, model_id, messages_Patient_assist, max_tokens, temperature_patient)
            print("\n Patient response: \n", response_content)
        else:  # Therapist's turn
            print("=" * 80)
            print("Tree utterance number: ", i)
            role_Patient = "user"
            role_Therapist = "assistant"
            # generate two responses for the therapist (branching)
            responses = generate_therapist_responses(therapist_model, therapis_tokenizer, messages_Therapist_assist, max_tokens, temperature_tree, 2)
            if responses is None:
                print("Could not generate the required number of valid responses. Skipping...")
                return None, None, None, None, None, None


            # handle SESSION ENDED
            for response in responses:
                if "SESSION ENDED" in response.upper():
                    print("Response with SESSION ENDED: ", response)
                    session_endded_by, session_endded_explanation, response = handle_session_end(response, i) # handle session ended
                    print("Response after handling SESSION ENDED: ", response)

            print("-" * 80)
            # print("Therapist responses: \n", responses)
            print("Therapist responses: ")
            for j, response in enumerate(responses):
                print(f"Response {j+1}: {response}")


            # split the conversation to two branches and continue each branch for look_ahead steps
            conversation_one, messages_Patient_assist_one, messages_Therapist_assist_one = prepare_lookahead_conversation(conversation, messages_Patient_assist, messages_Therapist_assist, responses[0])
            conversation_two, messages_Patient_assist_two, messages_Therapist_assist_two = prepare_lookahead_conversation(conversation, messages_Patient_assist, messages_Therapist_assist, responses[1])

            # continue the conversation for look_ahead steps
            print("-" * 40 + "Response one Look-Ahead" + "-" * 40)
            print("response one: ", responses[0])

            conversation_one, messages_Patient_assist_one, messages_Therapist_assist_one, _, _ = conversation_loop(
                conversation_one, messages_Patient_assist_one, messages_Therapist_assist_one, therapist_model, therapis_tokenizer, client, model_id, max_tokens, look_ahead, temperature_therapist, temperature_patient)
            print("-" * 40 + "Response two Look-Ahead" + "-" * 40)
            print("response two: ", responses[1])
            conversation_two, messages_Patient_assist_two, messages_Therapist_assist_two, _, _ = conversation_loop(
                conversation_two, messages_Patient_assist_two, messages_Therapist_assist_two, therapist_model, therapis_tokenizer, client, model_id, max_tokens, look_ahead, temperature_therapist, temperature_patient)

            if conversation_one is None or conversation_two is None:
                print("Could not generate the required number of valid responses for the look-ahead steps. Skipping...")
                return None, None, None, None, None, None

            ################################################################################################################
            # evaluate the two conversations
            score_one_df = evaluate_conversation_and_extract_scores(
                conversation=conversation_one, questionnaire_id=questionnaire_id, model=model_id,
                include_notes=include_notes, print_response=print_response, eval_temperature=temperature_eval)
            
            score_two_df = evaluate_conversation_and_extract_scores(
                conversation=conversation_two, questionnaire_id=questionnaire_id, model=model_id,
                include_notes=include_notes, print_response=print_response, eval_temperature=temperature_eval)

            if score_one_df is None or score_two_df is None:
                print("Could not generate the required number of valid scores for the conversations. Skipping...")
                return None, None, None, None, None, None

            print("-" * 80)
            print("score_one_final: ", score_one_df[column_name].values[0])
            print("score_two_final: ", score_two_df[column_name].values[0])

            # determine the winner and loser
            winning_response, losing_response, winning_scores_df, losing_scores_df, winning_conversation, losing_conversation = determine_winner(
                responses, score_one_df, score_two_df, conversation_one, conversation_two, column_name=column_name)
            print("-" * 80)
            print("winning_response: ", winning_response)
            print("losing_response: ", losing_response)
            print("-" * 80)
            
            # from a df with 1 row to a list
            winning_scores_dict = winning_scores_df.iloc[0].to_dict()
            losing_scores_dict = losing_scores_df.iloc[0].to_dict()
            

            # Append the preference data
            preference_data.append({
                "conversation": conversation.copy(),
                "messages": messages_Therapist_assist.copy(),
                "winning_response": winning_response,
                "losing_response": losing_response,
                "winning_scores_dict": winning_scores_dict,
                "losing_scores_dict": losing_scores_dict,
                "winning_conversation": winning_conversation,
                "losing_conversation": losing_conversation,
                "column_name": column_name
            })
            ################################################################################################################

            response_content = winning_response

        if "SESSION ENDED" in response_content.upper():
            session_endded_by, session_endded_explanation, response_content = handle_session_end(response_content, i)
            if response_content:
                update_conversation(conversation, messages_Patient_assist, messages_Therapist_assist, role_Patient, role_Therapist, response_content)
            break # End the conversation

        update_conversation(conversation, messages_Patient_assist, messages_Therapist_assist, role_Patient, role_Therapist, response_content)

    return preference_data, messages_Patient_assist, messages_Therapist_assist, conversation, session_endded_by, session_endded_explanation

Main Method

In [12]:
def synthesize_conversation_tree_lookAhead_for_permutation(
        system_prompt_therapist, 
        system_prompt_patient, 
        therapist_init_utterance, 
        patient_init_utterance, 
        therapist_model, 
        therapis_tokenizer, 
        client, 
        model_id="gpt-4o-mini-2024-07-18", 
        max_tokens=50, 
        num_init_utterances=2, 
        num_tree_utterances=6, 
        temperature_tree=1.2, 
        temperature_conversation_Therapist = 0.7, 
        temperature_conversation_Patient = 0.7, 
        temperature_eval=0.1, 
        questionnaire_id=3, 
        look_ahead=5,
        column_name: str = "WAI_TotalMean",
        include_notes: bool = False,
        print_response: bool = False
    ):
    # Create the initial conversation (no tree)
    init_conversation, init_messages_Patient_assist, init_messages_Therapist_assist, session_endded_by, session_endded_explanation = synthesize_conversation_therapistModel_patientOpenAI(
        system_prompt_therapist=system_prompt_therapist,
        system_prompt_patient=system_prompt_patient,
        therapist_init_utterance=therapist_init_utterance,
        patient_init_utterance=patient_init_utterance,
        therapist_model=therapist_model,
        therapis_tokenizer=therapis_tokenizer,
        client=client,
        model_id=model_id,
        max_tokens=max_tokens,
        num_utterances=num_init_utterances,
        temperature_therapist=temperature_conversation_Therapist,
        temperature_patient=temperature_conversation_Patient
    )

    if init_conversation is None:
        return None, None, None, None, None, None

    # Create the conversation tree (with look-ahead evaluation)
    preference_data, messages_Patient_assist, messages_Therapist_assist, conversation, session_endded_by, session_endded_explanation = synthesize_conversation_tree_lookAhead(
        init_messages_Patient_assist=init_messages_Patient_assist,
        init_messages_Therapist_assist=init_messages_Therapist_assist,
        init_conversation=init_conversation,
        therapist_model=therapist_model,
        therapis_tokenizer=therapis_tokenizer,
        client=client,
        model_id=model_id,
        max_tokens=max_tokens,
        num_utterances=num_tree_utterances,
        temperature_tree=temperature_tree,
        temperature_therapist=temperature_conversation_Therapist,
        temperature_patient=temperature_conversation_Patient,
        temperature_eval=temperature_eval,
        questionnaire_id=questionnaire_id,
        look_ahead=look_ahead,
        column_name=column_name,
        include_notes=include_notes,
        print_response=print_response
    )

    return preference_data, messages_Patient_assist, messages_Therapist_assist, conversation, session_endded_by, session_endded_explanation


## ===========================================================

### Generate permutations of patients

In [13]:
# get all permutations
permutations = generate_all_permutations(only_expert_therapist=True)
print(f"Number of permutations: {len(permutations)}")
print(f"perumtation keys: {permutations[0].keys()}")
# permutations is a list of dictionaries, where each dictionary has the following keys: 'counselor_init_utterance', 'counselor_system_prompt', 'patient_system_prompt', 'args'.
permutation_pd = pd.DataFrame(permutations)

Number of permutations: 96
perumtation keys: dict_keys(['counselor_init_utterance', 'counselor_system_prompt', 'patient_system_prompt', 'args'])


In [14]:
# Therapist Level for system prompts
good_level = CounselorPersonality.PersonalityLevel.Good
mediocre_level = CounselorPersonality.PersonalityLevel.Mediocre
bad_level = CounselorPersonality.PersonalityLevel.Bad
basic_level = CounselorPersonality.PersonalityLevel.BASIC
therapist = CounselorPersonality.choose_random_therapist_name()
# Therapist system prompt (good and bad)
therapist_good_system_prompt = CounselorPersonality.build_system_prompt(personality_level=good_level, name=therapist['name'])
therapist_mediocre_system_prompt = CounselorPersonality.build_system_prompt(personality_level=mediocre_level, name=therapist['name'])
therapist_bad_system_prompt = CounselorPersonality.build_system_prompt(personality_level=bad_level, name=therapist['name'])
therapist_basic_system_prompt = CounselorPersonality.build_system_prompt(personality_level=basic_level, name=therapist['name'])

# Therapist's initial utterance (good and bad), (for now using the good level)
therapist_good_init_utterance = CounselorPersonality.get_init_utterance(personality_level=good_level, name=therapist['name'])
therapist_mediocre_init_utterance = CounselorPersonality.get_init_utterance(personality_level=mediocre_level, name=therapist['name'])
therapist_bad_init_utterance = CounselorPersonality.get_init_utterance(personality_level=bad_level, name=therapist['name'])
therapist_basic_init_utterance = CounselorPersonality.get_init_utterance(personality_level=basic_level, name=therapist['name'])

In [ ]:
if False:
    print("Good level: ", good_level)
    print("Mediocre level: ", mediocre_level)
    print("Bad level: ", bad_level)
    print("Basic level: ", basic_level)
    print("-" * 80)
    print(f"Therapist good system prompt: \n{therapist_good_system_prompt}\n")
    print(f"Therapist mediocre system prompt: \n{therapist_mediocre_system_prompt}\n")
    print(f"Therapist bad system prompt: \n{therapist_bad_system_prompt}\n")
    print(f"Therapist basic system prompt: \n{therapist_basic_system_prompt}\n")
    print("-" * 80)
    print(f"Therapist good init utterance: \n{therapist_good_init_utterance}\n")
    print(f"Therapist mediocre init utterance: \n{therapist_mediocre_init_utterance}\n")
    print(f"Therapist bad init utterance: \n{therapist_bad_init_utterance}\n")
    print(f"Therapist basic init utterance: \n{therapist_basic_init_utterance}\n")

## ===========================================================

### Create Preference-Tree (Synthetic Data)

- Hyperparameters

In [16]:
# Model parameters
model_id = "gpt-4o-mini-2024-07-18" # Patient model and Eval model
max_tokens_per_response = 200 # 100, 200
num_init_utterances = 1 # 3, 1
num_tree_utterances = 41 # 41, 47
temperature_tree = 1.2 # 1.2, 1.4
temperature_conversation_Therapist = 0.9 # 0.7, 0.9
temperature_conversation_Patient = 0.7

# Parameters for usingEval pref_tree
# questionnaire_id = 3
temperature_eval = 0.1 # 0.1, 0.2
include_notes=True, # whether to include notes in the evaluation
print_response=False # whether to print the evaluation response

In [ ]:
path_to_save = f"LLM_DATA/Conversation_Trees_V3/{oracle_name}/LookAhead_{lookAhead}/TTree{temperature_tree}_TT{temperature_conversation_Therapist}_TP{temperature_conversation_Patient}_TE{temperature_eval}_V{version+1}"

path_to_save = f"/content/drive/MyDrive/{path_to_save}" # save to google drive

- synthesize_conversation_tree_for_permutation


In [ ]:
total_permutations = len(permutations)
count = 0

while count < total_permutations:
    count = 0

    for i in range(0, len(permutations)):
        print(f"Permutation {i}")
        print("=" * 160)

        # if saved file exists, skip
        if os.path.exists(f"{path_to_save}/pref_data_{i}.csv"):
            print(f"File exists: {path_to_save}/pref_data_{i}.csv")
            count += 1
            continue


        permutation = permutations[i]
        preference_data, messages_Patient_assist, messages_Therapist_assist, conversation, session_endded_by, session_endded_explanation = synthesize_conversation_tree_lookAhead_for_permutation(
            system_prompt_therapist=therapist_good_system_prompt, # system prompt for the therapis
            system_prompt_patient=permutation["patient_system_prompt"], # system prompt for the patient
            therapist_init_utterance=therapist_good_init_utterance, # therapist's initial utterance
            patient_init_utterance="", # patient's initial utterance
            therapist_model=therapist_model, # therapist model
            therapis_tokenizer=tokenizer, # therapist tokenizer
            client=client, # OpenAI client
            model_id=model_id, # model id to use
            max_tokens=max_tokens_per_response, # maximum tokens for each response
            num_init_utterances=num_init_utterances, # number of turns in the initial conversation (Not including the therapist's initial utterance)
            num_tree_utterances=num_tree_utterances, # number of turns in the conversation tree (Not including the therapist's and patient's initial utterances)
            temperature_tree=temperature_tree, # temperature for sampling
            temperature_conversation_Therapist=temperature_conversation_Therapist, # temperature for the therapist's responses
            temperature_conversation_Patient=temperature_conversation_Patient, # temperature for the patient's responses
            temperature_eval=temperature_eval, # temperature for evaluation
            questionnaire_id=questionnaire_id, # questionnaire id to use for evaluation
            look_ahead=lookAhead,
            column_name=final_column_name, # column name to use for determining the winner
            include_notes=include_notes, # whether to include notes in the evaluation
            print_response=print_response # whether to print the evaluation response
        )

        if preference_data is None:
            print("Could not generate the required number of valid responses. Skipping permutation number: ", i)
            continue

        pd_pref_data = pd.DataFrame(preference_data)
        print(pd_pref_data.info())
        print_conversation(conversation, max_width=80)
        print("Conversation Length: ", len(conversation))
        print(pd_pref_data.tail())
        print("=" * 160 + "\n")

        # save the preference data to a csv file
        pd_pref_data.to_csv(f"{path_to_save}/pref_data_{i}.csv", index=False)
        count += 1

    #pd_pref_data.head()

## ===========================================================

### Create Conversations and Evaluate them (Evaluation)
- for each permutation of patients synthesize a conversation and evaluate it using the Oracle

In [17]:
# Hyperparameters for the conversation
model_id = "gpt-4o-mini-2024-07-18"
max_tokens_per_response = 200 # 100, 200
num_uttrances = 49 # not including the therapist's initial utterance (so 50 turns in total)
temperature_theapist = 0.9 # 0.7, 1.0, 0.9
temperature_patient = 0.7
temperature_eval = 0.1
# questionnaire_id = 3
include_notes=True, # whether to include notes in the evaluation
print_response=False # whether to print the evaluation response

In [ ]:
##### Base #####
# path_to_save = f"LLM_DATA/Conversation_with_Eval_V2/Base/Good_50_TT{temperature_theapist}_TP{temperature_patient}_TE{temperature_eval}"

# ##### Look-Ahead #####
path_to_save = f"LLM_DATA/Conversation_with_Eval_V3/{oracle_name}/LookAhead_{lookAhead}/TTree{temperature_tree}_TT{temperature_theapist}_TP{temperature_patient}_TE{temperature_eval}_V{version}"

path_to_save = f"/content/drive/MyDrive/{path_to_save}" # save to google drive


Create Conversations

In [19]:
# create conversations
total_permutations = len(permutations)
count = 0

while count < total_permutations:
    count = 0

    for i in tqdm(range(0, len(permutations))):
    # iterate over all permutations
        print(f"Permutation {i}")

        if os.path.exists(f"{path_to_save}/conversation_{i}.csv"):
            print(f"File exists: {path_to_save}/conversation_{i}.csv")
            count += 1
            continue

        permutation = permutations[i]
        system_prompt_patient = permutation["patient_system_prompt"]
        print("=" * 160 + "\n")

        # create a conversation
        conversation, messages_Patient_assist, messages_Therapist_assist, session_endded_by, session_endded_explanation = synthesize_conversation_therapistModel_patientOpenAI(
            system_prompt_therapist=therapist_good_system_prompt, # system prompt for the therapist
            system_prompt_patient=system_prompt_patient, # system prompt for the patient
            therapist_init_utterance=therapist_good_init_utterance, # therapist's initial utterance
            patient_init_utterance="", # patient's initial utterance
            therapist_model=therapist_model, # therapist model
            therapis_tokenizer=tokenizer, # therapist tokenizer
            client=client, # OpenAI client
            model_id=model_id, # model id to use for the patient
            max_tokens=max_tokens_per_response, # maximum tokens for each response
            num_utterances=num_uttrances, # number of turns in the conversation (Not including the therapist's initial utterance)
            temperature_therapist=temperature_theapist, # temperature for the therapist's responses
            temperature_patient=temperature_patient # temperature for the patient's responses
        )

        if conversation is None:
            print("Could not generate the required number of valid responses. Skipping permutation number: ", i)
            continue

        print("Conversation Length: ", len(conversation))
        # save the conversation to a csv file
        pd_conversation = pd.DataFrame(
            {
                "conversation": conversation,
                "session_endded_by": session_endded_by,
                "session_endded_explanation": session_endded_explanation
            }
        )
        pd_conversation.to_csv(f"{path_to_save}/conversation_{i}.csv", index=False)
        conversation_str = reconstruct_conversation_text(conversation)   




100%|██████████| 96/96 [00:00<00:00, 37076.72it/s]

Permutation 0
File exists: LLM_DATA/Conversation_with_Eval_V3/WAI/LookAhead_0/TTree1.2_TT0.9_TP0.7_TE0.1_V5/conversation_0.csv
Permutation 1
File exists: LLM_DATA/Conversation_with_Eval_V3/WAI/LookAhead_0/TTree1.2_TT0.9_TP0.7_TE0.1_V5/conversation_1.csv
Permutation 2
File exists: LLM_DATA/Conversation_with_Eval_V3/WAI/LookAhead_0/TTree1.2_TT0.9_TP0.7_TE0.1_V5/conversation_2.csv
Permutation 3
File exists: LLM_DATA/Conversation_with_Eval_V3/WAI/LookAhead_0/TTree1.2_TT0.9_TP0.7_TE0.1_V5/conversation_3.csv
Permutation 4
File exists: LLM_DATA/Conversation_with_Eval_V3/WAI/LookAhead_0/TTree1.2_TT0.9_TP0.7_TE0.1_V5/conversation_4.csv
Permutation 5
File exists: LLM_DATA/Conversation_with_Eval_V3/WAI/LookAhead_0/TTree1.2_TT0.9_TP0.7_TE0.1_V5/conversation_5.csv
Permutation 6
File exists: LLM_DATA/Conversation_with_Eval_V3/WAI/LookAhead_0/TTree1.2_TT0.9_TP0.7_TE0.1_V5/conversation_6.csv
Permutation 7
File exists: LLM_DATA/Conversation_with_Eval_V3/WAI/LookAhead_0/TTree1.2_TT0.9_TP0.7_TE0.1_V5/con

Create Scores

In [ ]:
# create scores
total_permutations = len(permutations)
count = 0

while count < total_permutations:
    count = 0

    for i in tqdm(range(0, len(permutations))):
    # iterate over all permutations
        print(f"Permutation {i}")

        if os.path.exists(f"{path_to_save}/scores_{i}_Q1.csv"):
            print(f"File exists: {path_to_save}/scores_{i}_Q1.csv")
            count += 1
            continue

        # load the conversation
        pd_conversation = pd.read_csv(f"{path_to_save}/conversation_{i}.csv")
        conversation = pd_conversation["conversation"].tolist()

        # Evaluate conversation using the evaluation questionnaire
        print("Evaluating the conversation...")
        eval_df = evaluate_conversation_and_extract_scores(
            conversation=conversation, 
            model=model_id, 
            questionnaire_id=1, 
            include_notes=include_notes, 
            print_response=print_response, 
            eval_temperature=temperature_eval
        )
   

        if eval_df is None:
            print("Could not generate the required number of valid scores for the conversation. Skipping permutation number: ", i)
            continue

        # Print the evaluation results
        display(eval_df)

        # save the scores
        eval_df.to_csv(f"{path_to_save}/scores_{i}_Q1.csv", index=False)
        count += 1
        print("=" * 160 + "\n")

  0%|          | 0/96 [00:00<?, ?it/s]

Permutation 0
Evaluating the conversation...


  0%|          | 0/96 [00:06<?, ?it/s]


KeyboardInterrupt: 